# Spectral bias of PINNs — a multiscale ODE toy problem

Neural networks learn **low frequencies first and high frequencies (much) later** — the
*spectral bias* or *F-principle*. For PINNs this is a real failure mode: multiscale
solutions (turbulence, boundary layers, wave packets) have exactly the high-frequency
content a plain MLP refuses to learn.

**The toy problem** — the simplest ODE with two scales:
$$\frac{du}{dx} = f(x),\qquad u(0)=0,\qquad u_{exact}(x) = \underbrace{\sin(2\pi x)}_{slow,\ k=1} + \underbrace{0.1\,\sin(50\pi x)}_{fast,\ k=25}$$
so $f(x) = 2\pi\cos(2\pi x) + 5\pi\cos(50\pi x)$. One slow wave + small fast ripples.

**What you will see**
1. A standard tanh PINN captures the slow sine within ~500 epochs, then **stalls forever**
   on the ripples — the fast-mode amplitude stays at ~0 even after 6000 epochs.
2. The classic fix — **random Fourier features** — captures *both* scales in ~500 epochs.

Runs in well under a minute on Colab (CPU or GPU).

In [ ]:
# Cell 1 -- Problem setup and a spectral 'measuring stick'
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

PI = np.pi
A_LO, K_LO = 1.0, 2*PI      # slow mode:  sin(2 pi x),  k=1
A_HI, K_HI = 0.1, 50*PI     # fast mode:  0.1 sin(50 pi x),  k=25

def u_exact(x): return A_LO*torch.sin(K_LO*x) + A_HI*torch.sin(K_HI*x)
def f_rhs(x):   return A_LO*K_LO*torch.cos(K_LO*x) + A_HI*K_HI*torch.cos(K_HI*x)

# dense grid for evaluation + projection onto the two modes
xg = torch.linspace(0, 1, 2001, device=device).reshape(-1, 1)
sin_lo = torch.sin(K_LO*xg); sin_hi = torch.sin(K_HI*xg)

def mode_coeffs(up):
    """L2-projection amplitudes of the prediction onto the two sine modes.
    If the network has learned mode k with amplitude a, this returns ~a."""
    a_lo = 2*torch.trapz(up.ravel()*sin_lo.ravel(), xg.ravel()).item()
    a_hi = 2*torch.trapz(up.ravel()*sin_hi.ravel(), xg.ravel()).item()
    return a_lo, a_hi

plt.figure(figsize=(9, 3.2))
plt.plot(xg.cpu().ravel(), u_exact(xg).cpu().ravel(), 'g', lw=1.2)
plt.title('Target: slow wave (k=1) + fast ripples (k=25)')
plt.xlabel('x'); plt.ylabel('u'); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# Cell 2 -- The two contenders + a shared PINN training loop
class MLP(nn.Module):
    """Standard tanh PINN — will show spectral bias."""
    def __init__(self, h=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, h), nn.Tanh(),
                                 nn.Linear(h, h), nn.Tanh(),
                                 nn.Linear(h, h), nn.Tanh(),
                                 nn.Linear(h, 1))
    def forward(self, x): return self.net(x)

class FourierMLP(nn.Module):
    """The fix: map x -> [sin(2 pi B x), cos(2 pi B x)] first (random Fourier features).
    The random frequencies B ~ N(0, sigma^2) hand the network high-frequency
    'basis functions' for free, flattening the spectral bias."""
    def __init__(self, h=64, m=64, sigma=10.0):
        super().__init__()
        self.register_buffer('B', torch.randn(1, m) * sigma)
        self.net = nn.Sequential(nn.Linear(2*m, h), nn.Tanh(),
                                 nn.Linear(h, h), nn.Tanh(),
                                 nn.Linear(h, 1))
    def forward(self, x):
        z = 2*PI * x @ self.B
        return self.net(torch.cat([torch.sin(z), torch.cos(z)], 1))

def train(model, epochs=6000, tag=''):
    """PINN loss:  ||u'(x) - f(x)||^2  +  10*u(0)^2.   Tracks mode amplitudes."""
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    hist = {'epoch': [], 'a_lo': [], 'a_hi': [], 'l2': []}
    snaps = {}
    t0 = time.perf_counter()
    for e in range(epochs):
        opt.zero_grad()
        x = torch.rand(512, 1, device=device).requires_grad_(True)
        u = model(x)
        ux = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
        loss = ((ux - f_rhs(x))**2).mean() + 10*model(torch.zeros(1, 1, device=device))[0, 0]**2
        loss.backward(); opt.step()
        if e % 100 == 0 or e == epochs-1:
            with torch.no_grad():
                up = model(xg)
                a_lo, a_hi = mode_coeffs(up)
                hist['epoch'].append(e); hist['a_lo'].append(a_lo); hist['a_hi'].append(a_hi)
                hist['l2'].append(torch.sqrt(torch.mean((up - u_exact(xg))**2)).item())
        if e in (500, epochs-1):
            with torch.no_grad(): snaps[e] = model(xg).cpu().numpy().ravel()
    print(f'{tag}: trained {epochs} epochs in {time.perf_counter()-t0:.1f} s | '
          f'final a_lo={hist["a_lo"][-1]:.3f} (->1.0), a_hi={hist["a_hi"][-1]:.4f} (->0.1)')
    return hist, snaps

In [ ]:
# Cell 3 -- Train both with the SAME budget
torch.manual_seed(0)
hist_std, snaps_std = train(MLP(),        epochs=6000, tag='standard tanh MLP ')
torch.manual_seed(0)
hist_ff,  snaps_ff  = train(FourierMLP(), epochs=6000, tag='Fourier-feature MLP')

In [ ]:
# Cell 4 -- THE spectral-bias picture
xp = xg.cpu().numpy().ravel(); ue = u_exact(xg).cpu().numpy().ravel()
last = max(snaps_std)
fig, ax = plt.subplots(2, 2, figsize=(13, 7))

# top row: predictions vs exact
for j, (snaps, name) in enumerate([(snaps_std, 'standard MLP'), (snaps_ff, 'Fourier features')]):
    ax[0, j].plot(xp, ue, 'g', lw=1.0, alpha=.7, label='exact')
    ax[0, j].plot(xp, snaps[last], 'r', lw=1.0, label=f'PINN @ {last+1} epochs')
    ax[0, j].set_title(f'{name}: prediction vs exact')
    ax[0, j].set_xlabel('x'); ax[0, j].legend(); ax[0, j].grid(alpha=.3)

# bottom left: learned mode amplitudes vs epoch  (the money plot)
ax[1, 0].plot(hist_std['epoch'], hist_std['a_lo'], 'b-',  label='slow mode k=1 (std)')
ax[1, 0].plot(hist_std['epoch'], hist_std['a_hi'], 'b--', label='fast mode k=25 (std)')
ax[1, 0].plot(hist_ff['epoch'],  hist_ff['a_lo'],  'r-',  label='slow mode k=1 (FF)')
ax[1, 0].plot(hist_ff['epoch'],  hist_ff['a_hi'],  'r--', label='fast mode k=25 (FF)')
ax[1, 0].axhline(1.0, color='gray', ls=':', lw=.8); ax[1, 0].axhline(0.1, color='gray', ls=':', lw=.8)
ax[1, 0].set_xlabel('epoch'); ax[1, 0].set_ylabel('learned amplitude')
ax[1, 0].set_title('Spectral bias: fast mode stalls at 0 for the standard MLP')
ax[1, 0].legend(fontsize=8); ax[1, 0].grid(alpha=.3)

# bottom right: L2 error vs epoch
ax[1, 1].semilogy(hist_std['epoch'], hist_std['l2'], 'b', label='standard MLP')
ax[1, 1].semilogy(hist_ff['epoch'],  hist_ff['l2'],  'r', label='Fourier features')
ax[1, 1].axhline(A_HI/np.sqrt(2), color='gray', ls=':', label='fast-mode floor 0.1/sqrt(2)')
ax[1, 1].set_xlabel('epoch'); ax[1, 1].set_ylabel('L2 error')
ax[1, 1].set_title('Error plateaus exactly at the unlearned fast mode')
ax[1, 1].legend(fontsize=9); ax[1, 1].grid(alpha=.3, which='both')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 5 -- Zoom in: the ripples the standard PINN never learns
m = (xp >= 0.2) & (xp <= 0.4)
plt.figure(figsize=(11, 3.6))
plt.plot(xp[m], ue[m], 'g', lw=1.5, label='exact (has ripples)')
plt.plot(xp[m], snaps_std[last][m], 'b--', lw=1.5, label='standard MLP (smooth only)')
plt.plot(xp[m], snaps_ff[last][m], 'r:',  lw=1.8, label='Fourier features (ripples!)')
plt.xlabel('x'); plt.ylabel('u'); plt.legend(); plt.grid(alpha=.3)
plt.title('Zoom on x in [0.2, 0.4] after the same training budget')
plt.tight_layout(); plt.show()

## Takeaways

- **Spectral bias is real and dramatic.** With identical budgets, the standard tanh PINN
  learns the $k=1$ mode in ~500 epochs and *never moves* the $k=25$ amplitude off zero —
  its L2 error plateaus at exactly the energy of the unlearned mode ($0.1/\sqrt{2}\approx0.07$).
- **Why:** in the NTK picture, gradient descent's convergence rate for mode $k$ decays
  rapidly with $k$ — high frequencies correspond to tiny NTK eigenvalues, so their
  training is orders of magnitude slower.
- **The fix costs one line:** random Fourier features $x \mapsto [\sin(2\pi Bx),\cos(2\pi Bx)]$
  hand the network high-frequency basis functions up front. Both modes converge in ~500 epochs.
- **Why PINN users must care:** boundary layers, turbulence spectra, wave propagation and
  shocks are all high-frequency. This is a core reason vanilla PINNs struggle on stiff or
  multiscale problems — and why libraries (e.g. underPINN's `FourierMLP`) ship Fourier
  networks as a first-class option.

**Experiments to try:** raise the fast mode to $k=50$ (does FF with $\sigma=10$ still work?
tune $\sigma$); give the standard MLP 60,000 epochs (it eventually creeps up — slowly);
change the fast amplitude to 1.0; try `nn.SiLU()`.